# SUPATLANTIQUE Dataset & Digital Image Forensics

## Week 1: Dataset Understanding & Conceptual Analysis

**Goal**: Understand the dataset structure, scanner identification workflow, and forgery detection before coding.

---


## 1. Dataset Overview

### What is SUPATLANTIQUE?

The SUPATLANTIQUE is a comprehensive scanned documents database designed for:

- **Scanner Source Identification** (SSI)
- **Forgery/Tamper Detection**
- **Digital Image Forensics Research**

### Key Characteristics:

| Aspect         | Details                                                    |
| -------------- | ---------------------------------------------------------- |
| **Purpose**    | Evaluate scanner fingerprints and document tampering       |
| **Resolution** | 150 DPI & 300 DPI                                          |
| **Scanners**   | Multiple brands (Canon, HP, Xerox, Brother, etc.)          |
| **Images**     | Over 2,000+ original scans                                 |
| **Categories** | Multiple tampering types (copy-move, splicing, retouching) |

---


## 2. Dataset Structure

### Folder Organization:

```
SUPATLANTIQUE/
├── Flatfield/
│   ├── canon_lide_120_150.png
│   ├── canon_lide_120_300.png
│   ├── hp_scanner_150.png
│   └── ...
│
├── Originals/
│   ├── canon_lide_120/
│   │   ├── document_001_150.jpg
│   │   ├── document_001_300.jpg
│   │   └── ...
│   ├── hp_scanner/
│   └── ...
│
├── Official/
│   ├── canon_lide_120/
│   └── ...
│
├── Wikipedia/
│   └── ... (scans of Wikipedia articles)
│
├── Original_PDFs/
│   ├── document_001.pdf
│   └── ...
│
└── Tampered_images/
    ├── copy_move/
    ├── splicing/
    ├── retouching/
    └── (with corresponding masks)
```

---


## 3. Understanding Each Folder Type

### 3.1 Flatfield Images

- **Purpose**: Reference images for scanner calibration
- **Content**: Blank/uniform scans from each scanner
- **Use**: Extract raw scanner fingerprints without document content interference
- **Why Important**: Scanner introduces unique noise patterns visible in blank scans

```python
# Flatfield shows scanner-specific patterns:
# - Dust particles on scanner bed
# - CCD/sensor artifacts
# - Optical aberrations
# - Electrical noise
```

### 3.2 Original Scans (Originals folder)

- **Purpose**: Groundtruth scans of documents
- **Content**: Real document pages scanned at different resolutions
- **Attributes**:
  - Multiple documents per scanner
  - 150 DPI and 300 DPI versions
  - Known scanner source (ground truth)
- **Use**: Train scanner identification models

### 3.3 Official Scans

- **Purpose**: Reference/official versions of documents
- **Content**: High-quality baseline scans
- **Use**: Compare with tampered versions

### 3.4 Wikipedia Scans

- **Purpose**: Additional authentic scans from alternative source
- **Content**: Wikipedia articles scanned with various scanners
- **Use**: Test generalization across different document sources

### 3.5 Original PDFs

- **Purpose**: Source documents before scanning
- **Content**: PDF files that were printed and then scanned
- **Use**: Understand document content, synthetic tampering generation

### 3.6 Tampered Images

- **Purpose**: Images with intentional modifications
- **Types**:
  1. **Copy-Move**: Region copied and pasted within same image
  2. **Splicing**: Content from different images combined
  3. **Retouching**: Content modified/removed using editing tools
- **Includes**: Masks showing exact tampered regions

---


## 4. Scanner Identification Hierarchies

### Three Levels of Identification:

#### Level 1: Brand Identification

```
Scanner → Canon? | HP? | Xerox? | Brother?
Accuracy: Usually 95%+
Reason: Different optical systems, sensor types
```

#### Level 2: Model Identification

```
Canon → LiDe 120? | LiDe 220? | CanoScan?
Accuracy: Usually 80-90%
Reason: Similar hardware, different firmware/calibration
```

#### Level 3: Instance Identification

```
Canon LiDe 120 #1 → vs Canon LiDe 120 #2?
Accuracy: Usually 60-75%
Reason: Minimal differences between individual units
Challenge: Requires very sensitive fingerprint extraction
```

**Note**: SUPATLANTIQUE focuses on **Brand and Model** identification (Levels 1-2)

---


## 5. How Scanner Identification Works

### The Core Concept: Scanner Fingerprinting

```
Every scanner leaves unique traces in every image it scans:

Scanned Document = Document Content + Scanner Fingerprint
```

### The Scanner Fingerprint Includes:

1. **Photo Response Non-Uniformity (PRNU)**
   - CCD/sensor sensitivity varies pixel-to-pixel
   - Temporal noise components
   - Pattern repeats across all images from same scanner

2. **Fixed Pattern Noise**
   - Dark current variations
   - Read noise patterns
   - Consistent across images

3. **Optical Aberrations**
   - Vignetting (darker at edges)
   - Distortion patterns
   - Chromatic aberration

4. **Geometric Artifacts**
   - Dust/scratches on scanner bed
   - Misalignment patterns
   - Sensor defects

### Extraction Workflow:

```
Step 1: Load Image
   ↓
Step 2: Convert to Grayscale/Extract Color Channel
   ↓
Step 3: High-Pass Filter (Remove content, keep noise)
   ↓
Step 4: Analyze Residual in Frequency Domain (FFT/DCT)
   ↓
Step 5: Extract Features (Statistical, Frequency-based)
   ↓
Step 6: Feed to Classifier (SVM, RF, Neural Network)
   ↓
Step 7: Output: Scanner Brand/Model Prediction + Confidence
```

---


## 6. How Forgery Detection Works

### Core Principle:

```
Tampered regions have different scanner fingerprint patterns
because they came from different sources or were modified.
```

### Three Tampering Categories:

#### 1. Copy-Move Forgery

- **What**: Region copied from one location to another within same image
- **Pattern**: Duplicated scanner fingerprint + minimal changes
- **Detection**: Find blocks with identical/similar fingerprints

#### 2. Splicing

- **What**: Content from different images combined
- **Pattern**: Different scanner fingerprints in different regions
- **Detection**: Find regions with inconsistent fingerprints

#### 3. Retouching

- **What**: Content modified using editing tools (erasing, cloning, etc.)
- **Pattern**: Missing or altered fingerprint patterns
- **Detection**: Find regions with disrupted fingerprint structure

### Detection Workflow:

```
Step 1: Load Suspected Image + Mask (if available)
   ↓
Step 2: Partition into Blocks (8x8 or 16x16 patches)
   ↓
Step 3: Extract Fingerprint from Each Block
   ↓
Step 4: Compute Similarity Between Blocks
   ↓
Step 5: Identify Inconsistent Regions
   ↓
Step 6: Compare with Ground Truth Mask
   ↓
Step 7: Output: Tampered Regions + Confidence Map
```

---


## 7. Image Processing & Analysis Fundamentals

### 7.1 Residual Extraction

```python
# High-pass filtering to remove content and keep noise

from scipy import ndimage
import numpy as np
from scipy.ndimage import gaussian_filter

# Method 1: Laplacian Filter
residual = image - gaussian_filter(image, sigma=1.5)

# Method 2: High-Pass Filter
kernel = np.array([[-1, -1, -1],
                     [-1, 8, -1],
                     [-1, -1, -1]])
residual = cv2.filter2D(image, -1, kernel)
```

### 7.2 Feature Extraction Types

**Statistical Features**:

- Mean, Variance, Skewness, Kurtosis
- Quantifies noise distribution

**Frequency Domain Features**:

- FFT magnitude, DCT coefficients
- Identifies scanner-specific frequency patterns

**Texture Features**:

- GLCM (Gray-Level Co-occurrence Matrix)
- Local Binary Patterns (LBP)
- Captures fingerprint structure

**Wavelet Features**:

- Decomposition using DWT
- Multi-scale fingerprint analysis

---


## 8. Resolution Impact (150 DPI vs 300 DPI)

### Key Differences:

| Aspect                      | 150 DPI          | 300 DPI        |
| --------------------------- | ---------------- | -------------- |
| **Image Size**              | Smaller          | Larger (~4x)   |
| **Fingerprint Visibility**  | Less detailed    | More detailed  |
| **Feature Extraction**      | Coarser patterns | Finer patterns |
| **Computational Cost**      | Lower            | Higher         |
| **Identification Accuracy** | ~85-90%          | ~90-95%        |

### Strategy:

- Train separate models for each resolution
- OR normalize to common resolution
- OR extract resolution-invariant features

---


## 9. Complete Workflow Overview

### For Scanner Identification:

```
┌─────────────────────────────────────────────────────────────────────┐
│ RAW IMAGE (JPG/PNG)                                                 │
└──────────────────────────┬──────────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────────┐
│ PREPROCESSING                                                       │
│ • Load image                                                        │
│ • Convert to grayscale (if needed)                                 │
│ • Normalize pixel values [0, 1] or [-1, 1]                         │
└──────────────────────────┬──────────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────────┐
│ FINGERPRINT EXTRACTION                                              │
│ • Apply high-pass filter (Laplacian/Wiener/Hann filter)            │
│ • Remove document content, keep scanner noise                       │
└──────────────────────────┬──────────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────────┐
│ FEATURE EXTRACTION                                                  │
│ • Statistical (mean, var, skewness, kurtosis)                      │
│ • Frequency domain (FFT/DCT magnitude)                              │
│ • Texture (GLCM, LBP)                                              │
│ • Wavelet coefficients                                              │
│ • Result: Feature Vector (100-500 dimensions)                       │
└──────────────────────────┬──────────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────────┐
│ CLASSIFICATION                                                      │
│ • SVM (multiclass)                                                  │
│ • Random Forest                                                     │
│ • Neural Network / CNN                                              │
│ • Input: Feature vector                                             │
│ • Output: Scanner predictions + confidence scores                   │
└──────────────────────────┬──────────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────────┐
│ OUTPUT                                                              │
│ • Top 3 Scanner Predictions                                         │
│ • Confidence Scores                                                 │
│ • Decision Confidence Threshold                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### For Forgery Detection:

```
┌──────────────────────────────────────────────────────────────────────┐
│ SUSPECTED IMAGE + OPTIONAL MASK                                      │
└──────────────────────┬───────────────────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────────────────────────────┐
│ PREPROCESSING & BLOCKING                                             │
│ • Partition into overlapping blocks (8x8 or 16x16)                  │
│ • Stride: usually 4 of 8 pixels                                     │
└──────────────────────┬───────────────────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────────────────────────────┐
│ FINGERPRINT EXTRACTION (per block)                                   │
│ • High-pass filter on each block                                    │
│ • Residual for each patch                                           │
└──────────────────────┬───────────────────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────────────────────────────┐
│ CONSISTENCY ANALYSIS                                                 │
│ • Compare fingerprints between adjacent blocks                       │
│ • Compute similarity/correlation                                    │
│ • Identify inconsistent regions                                     │
└──────────────────────┬───────────────────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────────────────────────────┐
│ FORGERY DETECTION OUTPUT                                             │
│ • Tampered Region Map                                               │
│ • Confidence Map (per pixel)                                        │
│ • Comparison with Ground Truth Mask                                 │
│ • Metrics: Precision, Recall, F1-Score                              │
└──────────────────────────────────────────────────────────────────────┘
```

---


## 10. Technology Stack Summary

### Image Processing

```python
import cv2              # OpenCV - core image ops
import numpy as np      # NumPy - numerical arrays
from PIL import Image   # Pillow - image I/O
from skimage import *   # scikit-image - image algorithms
```

### Frequency & Signal Processing

```python
from scipy import signal, fft, ndimage
import numpy.fft as fft
import pywt             # PyWavelets - wavelet transforms
```

### Machine Learning

```python
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
```

### Visualization

```python
import matplotlib.pyplot as plt
import seaborn as sns
```

### Deep Learning (Optional)

```python
import tensorflow as tf
from tensorflow import keras
```

---


## 11. Key Concepts to Remember

### Scanner Fingerprint

✓ Every scanner leaves unique traces (PRNU, noise patterns, optical artifacts)  
✓ These patterns are present in every image the scanner produces  
✓ Fingerprints are invisible to humans but detectable by algorithms

### Residual

✓ Residual = Image - High-Pass Filtered Image  
✓ Contains scanner noise, removes document content  
✓ Crucial intermediate step for fingerprint analysis

### Feature Vector

✓ Compact representation of fingerprint characteristics  
✓ Generated from residual using statistical/frequency analysis  
✓ Input to machine learning classifiers

### Forgery Types

✓ Copy-Move: Duplication within image (same fingerprint)  
✓ Splicing: Multiple sources (different fingerprints)  
✓ Retouching: Modification by editing tools (broken fingerprint)

### Ground Truth

✓ Known scanner source for Original images  
✓ Tampered region masks for forgery detection  
✓ Essential for training and evaluation

---


## 12. Learning Path for This Week

### Day 1-2: Dataset Familiarization

- [ ] Explore folder structure
- [ ] Understand image organization
- [ ] Load and visualize sample images from each category
- [ ] Compare flatfield images across scanners

### Day 3-4: Conceptual Understanding

- [ ] Study scanner fingerprint theory
- [ ] Understand residual extraction
- [ ] Learn feature extraction methods
- [ ] Understand forgery detection principles

### Day 5: Preliminary Analysis

- [ ] Write code to load and visualize data
- [ ] Extract and compare residuals from different scanners
- [ ] Visualize frequency domain differences
- [ ] Document findings

### By End of Week

✓ Can explain dataset structure clearly  
✓ Understand how scanner fingerprints work  
✓ Can outline complete preprocessing → classification workflow  
✓ Ready to start building feature extraction code

---


## 13. Questions to Reflect On

1. **Why are flatfield images important?**
   - Answer: They show raw scanner fingerprint without document content interference

2. **What makes scanners identifiable?**
   - Answer: Unique combinations of hardware (sensor, optics), firmware, and imperfections

3. **How are forgeries different from originals from fingerprint perspective?**
   - Answer: Inconsistent/missing fingerprints in tampered regions

4. **Why does resolution matter?**
   - Answer: Higher resolution captures finer fingerprint details, improving accuracy

5. **What features best capture scanner fingerprints?**
   - Answer: Combination of statistical (variance patterns) and frequency-domain features

6. **Can you identify individual scanner units vs just scanner models?**
   - Answer: Yes, but harder (requires instance-level fingerprinting)

---


## Summary

### This Week's Goal: Conceptual Clarity ✓

You now understand:

1. **Dataset Structure**: Folders contain specific image types for specific purposes
2. **Scanner Identification**: How unique fingerprints enable scanner identification
3. **Forgery Detection**: How inconsistent fingerprints reveal tampering
4. **Workflow**: Complete pipeline from raw image to scanner/forgery prediction
5. **Technology**: Tools and methods for each stage

### Next Week: Implementation

With this foundation, you'll be ready to code:

- Data loading and preprocessing
- Residual extraction algorithms
- Feature extraction functions
- Model training and evaluation

---

**Study Note**: Revisit this notebook frequently. Conceptual clarity is your foundation for successful implementation.
